# Evaluating Clusters & Choosing k

**Companion lesson:** https://ml-viz.vercel.app/courses/clustering/03-evaluating-clusters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'
np.random.seed(42)

## K-Means + the two classic diagnostics

Three true blobs; we let k range from 2 to 8 and see which k the metrics point at.

In [ ]:
def kmeans(X, k, iters=50):
    C = X[np.random.choice(len(X), k, replace=False)]
    for _ in range(iters):
        lbl = np.argmin(((X[:, None] - C[None]) ** 2).sum(-1), axis=1)
        C = np.array([X[lbl == j].mean(0) if (lbl == j).any() else C[j] for j in range(k)])
    inertia = ((X - C[lbl]) ** 2).sum()
    return lbl, C, inertia

blobs = [np.random.randn(60, 2) * 0.7 + c for c in [(-3, 0), (3, 2), (1, -3)]]
X = np.vstack(blobs)

## Silhouette, from its definition

In [ ]:
def silhouette(X, lbl):
    D = np.sqrt(((X[:, None] - X[None]) ** 2).sum(-1))
    s = np.zeros(len(X))
    for i in range(len(X)):
        own = lbl == lbl[i]; own[i] = False
        a = D[i, own].mean() if own.any() else 0
        b = min(D[i, lbl == j].mean() for j in set(lbl) if j != lbl[i])
        s[i] = (b - a) / max(a, b)
    return s

ks = range(2, 9); inertias, sils = [], []
for k in ks:
    lbl, C, inertia = kmeans(X, k)
    inertias.append(inertia); sils.append(silhouette(X, lbl).mean())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(ks), inertias, 'o-', color='#6366f1'); axes[0].set_title('inertia (elbow)')
axes[1].plot(list(ks), sils, 'o-', color='#14b8a6'); axes[1].set_title('mean silhouette')
for ax in axes: ax.set_xlabel('k'); ax.axvline(3, color='#f43f5e', ls=':', lw=1)
plt.tight_layout(); plt.show()
# inertia bends at k=3; silhouette peaks at k=3 — both agree here

## When the metrics lie: two moons

In [ ]:
t = np.linspace(0, np.pi, 100)
moons = np.vstack([np.c_[np.cos(t), np.sin(t)],
                   np.c_[1 - np.cos(t), 0.5 - np.sin(t)]]) + 0.07 * np.random.randn(200, 2)
true_lbl = np.array([0] * 100 + [1] * 100)

km_lbl, _, _ = kmeans(moons, 2)
print('silhouette of TRUE moon labels :', round(silhouette(moons, true_lbl).mean(), 3))
print('silhouette of K-Means labels   :', round(silhouette(moons, km_lbl).mean(), 3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, l, name in [(axes[0], true_lbl, 'true clusters'), (axes[1], km_lbl, 'K-Means k=2')]:
    ax.scatter(*moons.T, c=np.where(l == 0, '#6366f1', '#14b8a6'), s=12); ax.set_title(name)
plt.tight_layout(); plt.show()
# K-Means' wrong vertical split can SCORE HIGHER than the true crescents:
# silhouette rewards compact round blobs — the same bias K-Means has

**Try it:** write the stability check — run `kmeans` 20 times on random 80% subsets and count how often each pair of points co-clusters. At k=3 on the blobs, co-clustering rates are near 0 or 1 (stable); at k=5 they smear (unstable).